# Mouse FF GO follow-up — ShinyGO-style views

This notebook rebuilds the **FF up/down GO follow-up** from the underlying g:Profiler API so we can get closer to the ShinyGO outputs we actually care about: a ranked chart, a tree-like redundancy view, a network view, a genes view, and grouped term summaries.

The focus stays on **new work only**: the main side-specific FF branch, split by direction, with the strongest visuals centered on the upregulated side because that is where the cleaner signal sits.

In [ ]:
from pathlib import Path
import csv
import json
import math
import re
import urllib.request

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.cluster.hierarchy import dendrogram, linkage, leaves_list
from scipy.spatial.distance import squareform

sns.set_theme(style='whitegrid', context='talk')

ROOT = Path('/Users/pitergarcia/DataScience/Semester5/BIOL550/group_project/mouse_new')
FF_DIR = ROOT / 'differential_expression_all20' / 'derived_analysis' / 'ipsi_vs_contra_in_ff'
OUTDIR = FF_DIR / 'shinygo_style'
OUTDIR.mkdir(exist_ok=True)

selected = pd.read_csv(FF_DIR / 'selected_genes_bendpoint.tsv', sep='	')
symbols = pd.read_csv(FF_DIR / 'selected_genes_bendpoint_gene_symbols.tsv', sep='	').drop_duplicates()
symbol_map = dict(zip(symbols['gene_id'], symbols['symbol']))
anchor = pd.read_csv(FF_DIR / 'anchor_genes_up_down.tsv', sep='	')

up_ids = selected.loc[selected['log2FoldChange'] > 0, 'gene_id'].tolist()
down_ids = selected.loc[selected['log2FoldChange'] < 0, 'gene_id'].tolist()

display(pd.DataFrame({
    'Direction': ['Upregulated', 'Downregulated'],
    'Genes in bend-point core': [len(up_ids), len(down_ids)]
}))


**What we did:** Loaded the FF bend-point-selected genes, split them by direction, and set up a fresh output folder for the richer GO follow-up.

**What it shows:** We are not reusing the old summary tables blindly; we are rebuilding the GO layer from the selected genes themselves.

**Why it matters:** This gives us a real chance to produce outputs that look and behave more like ShinyGO instead of relying on rough visual analogies.